In [ ]:
# ============================================================
# FEATURE ENGINEERING & PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

# ============================================================
# LOAD DATA
# ============================================================

fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

# ============================================================
# DATETIME CONVERSION
# ============================================================

fraud_df["signup_time"] = pd.to_datetime(
    fraud_df["signup_time"]
)

fraud_df["purchase_time"] = pd.to_datetime(
    fraud_df["purchase_time"]
)

# ============================================================
# IP CONVERSION
# ============================================================

fraud_df["ip_address"] = (
    fraud_df["ip_address"]
    .astype(np.int64)
)

ip_df["lower_bound_ip_address"] = (
    ip_df["lower_bound_ip_address"]
    .astype(np.int64)
)

ip_df["upper_bound_ip_address"] = (
    ip_df["upper_bound_ip_address"]
    .astype(np.int64)
)

# ============================================================
# GEOLOCATION MERGE
# ============================================================

fraud_df = fraud_df.sort_values(
    "ip_address"
)

ip_df = ip_df.sort_values(
    "lower_bound_ip_address"
)

merged_df = pd.merge_asof(
    fraud_df,
    ip_df,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

merged_df = merged_df[
    merged_df["ip_address"]
    <= merged_df["upper_bound_ip_address"]
]

print("Merged Shape:")
print(merged_df.shape)

# ============================================================
# COUNTRY FRAUD ANALYSIS
# ============================================================

country_fraud = (
    merged_df
    .groupby("country")["class"]
    .mean()
    .sort_values(ascending=False)
)

print("\nTop Countries by Fraud Rate")
print(country_fraud.head(20))

# ============================================================
# FEATURE ENGINEERING
# ============================================================

merged_df["time_since_signup"] = (
    merged_df["purchase_time"]
    - merged_df["signup_time"]
).dt.total_seconds()

merged_df["hour_of_day"] = (
    merged_df["purchase_time"]
    .dt.hour
)

merged_df["day_of_week"] = (
    merged_df["purchase_time"]
    .dt.dayofweek
)

merged_df["transaction_count"] = (
    merged_df.groupby("user_id")
    ["user_id"]
    .transform("count")
)

merged_df["device_transaction_count"] = (
    merged_df.groupby("device_id")
    ["device_id"]
    .transform("count")
)

print("\nFeature Engineering Completed")

# ============================================================
# ONE HOT ENCODING
# ============================================================

merged_df = pd.get_dummies(
    merged_df,
    columns=[
        "source",
        "browser",
        "sex",
        "country"
    ],
    drop_first=True
)

# ============================================================
# DROP UNUSED COLUMNS
# ============================================================

drop_cols = [
    "signup_time",
    "purchase_time",
    "device_id"
]

for col in drop_cols:
    if col in merged_df.columns:
        merged_df.drop(
            col,
            axis=1,
            inplace=True
        )

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X = merged_df.drop(
    "class",
    axis=1
)

y = merged_df["class"]

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )
)

# ============================================================
# SCALING
# ============================================================

scaler = StandardScaler()

numeric_columns = X_train.select_dtypes(
    include=np.number
).columns

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

# ============================================================
# BEFORE SMOTE
# ============================================================

print("\nBefore SMOTE")

print(y_train.value_counts())

# ============================================================
# SMOTE
# ============================================================

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

# ============================================================
# AFTER SMOTE
# ============================================================

print("\nAfter SMOTE")

print(y_train_smote.value_counts())

# ============================================================
# SAVE DATASETS
# ============================================================

merged_df.to_csv(
    "../data/processed/fraud_data_engineered.csv",
    index=False
)

print("\nProcessed dataset saved.")

print("\nPREPROCESSING COMPLETED SUCCESSFULLY")